# Step 1: Install the required libraries


In [2]:
pip install statsbombpy pandas numpy sqlalchemy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


# Step 2: Explore what data is available


In [3]:
from statsbombpy import sb

# See all available competitions
competitions = sb.competitions()
print(competitions[['competition_name', 'season_name', 'competition_id', 'season_id']].to_string())

/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


           competition_name season_name  competition_id  season_id
0             1. Bundesliga   2023/2024               9        281
1             1. Bundesliga   2015/2016               9         27
2    African Cup of Nations        2023            1267        107
3          Champions League   2018/2019              16          4
4          Champions League   2017/2018              16          1
5          Champions League   2016/2017              16          2
6          Champions League   2015/2016              16         27
7          Champions League   2014/2015              16         26
8          Champions League   2013/2014              16         25
9          Champions League   2012/2013              16         24
10         Champions League   2011/2012              16         23
11         Champions League   2010/2011              16         22
12         Champions League   2009/2010              16         21
13         Champions League   2008/2009              16       

# Step 3: Pick a competition to build on


In [4]:
# Filter for La Liga
laliga = competitions[competitions['competition_name'] == 'La Liga']
print(laliga[['season_name', 'competition_id', 'season_id']])

   season_name  competition_id  season_id
38   2020/2021              11         90
39   2019/2020              11         42
40   2018/2019              11          4
41   2017/2018              11          1
42   2016/2017              11          2
43   2015/2016              11         27
44   2014/2015              11         26
45   2013/2014              11         25
46   2012/2013              11         24
47   2011/2012              11         23
48   2010/2011              11         22
49   2009/2010              11         21
50   2008/2009              11         41
51   2007/2008              11         40
52   2006/2007              11         39
53   2005/2006              11         38
54   2004/2005              11         37
55   1973/1974              11        278


# Step 4: Pull all matches for La Liga 2015/16


In [5]:
# Pull all matches
matches = sb.matches(competition_id=11, season_id=27)
print(f"Total matches: {len(matches)}")
print(matches[['match_id', 'match_date', 'home_team', 'away_team', 'home_score', 'away_score']].head(10))

/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Total matches: 380
   match_id  match_date               home_team    away_team  home_score  \
0   3825848  2015-09-23              Levante UD        Eibar           2   
1   3825895  2015-09-23              Las Palmas      Sevilla           2   
2   3825894  2016-05-01  RC Deportivo La Coruña       Getafe           0   
3   3825855  2016-05-02                  Málaga   Levante UD           3   
4   3825908  2016-05-15                Espanyol        Eibar           4   
5   3825883  2016-05-15                  Málaga   Las Palmas           4   
6   3825900  2016-05-15          Sporting Gijón   Villarreal           2   
7   3825902  2016-05-15          Rayo Vallecano   Levante UD           3   
8   3825876  2016-05-15              Real Betis       Getafe           2   
9   3825846  2016-05-14  RC Deportivo La Coruña  Real Madrid           0   

   away_score  
0           2  
1           0  
2           2  
3           1  
4           2  
5           1  
6           0  
7           1  


# Step 5: Pull events for ONE match first (test run)


In [6]:
# Grab first match_id
sample_match_id = matches['match_id'].iloc[0]

# Pull events for that match
events = sb.events(match_id=sample_match_id)
print(f"Total events in this match: {len(events)}")
print(events['type'].value_counts())  # See what event types exist

/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Total events in this match: 2614
type
Pass               910
Ball Receipt*      401
Carry              400
Pressure           291
Ball Recovery      107
Duel                91
Clearance           57
Block               49
Dribble             42
Foul Committed      38
Miscontrol          36
Foul Won            36
Goal Keeper         31
Dribbled Past       29
Interception        26
Shot                22
Dispossessed        20
Substitution         6
Injury Stoppage      5
Half End             4
Half Start           4
Tactical Shift       3
Starting XI          2
Player On            1
Player Off           1
Bad Behaviour        1
Shield               1
Name: count, dtype: int64


# Step 6: Build the SQLite Database Schema


In [7]:
import sqlite3
import pandas as pd

# Create DB
conn = sqlite3.connect('/kaggle/working/clubiq.db')
cursor = conn.cursor()

# Table 1: matches
cursor.execute('''
    CREATE TABLE IF NOT EXISTS matches (
        match_id INTEGER PRIMARY KEY,
        match_date TEXT,
        home_team TEXT,
        away_team TEXT,
        home_score INTEGER,
        away_score INTEGER,
        result TEXT
    )
''')

# Table 2: players
cursor.execute('''
    CREATE TABLE IF NOT EXISTS players (
        player_id TEXT PRIMARY KEY,
        player_name TEXT,
        team TEXT,
        position TEXT
    )
''')

# Table 3: events
cursor.execute('''
    CREATE TABLE IF NOT EXISTS events (
        event_id TEXT PRIMARY KEY,
        match_id INTEGER,
        player_id TEXT,
        player_name TEXT,
        team TEXT,
        event_type TEXT,
        minute INTEGER,
        second INTEGER,
        location_x REAL,
        location_y REAL,
        outcome TEXT
    )
''')

conn.commit()
print("✅ Database and tables created successfully!")

✅ Database and tables created successfully!


# Step 7: Populate the matches table


In [8]:
# Add result column
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 'home_win'
    elif row['home_score'] < row['away_score']:
        return 'away_win'
    else:
        return 'draw'

matches['result'] = matches.apply(get_result, axis=1)

# Insert into DB
matches[['match_id','match_date','home_team','away_team','home_score','away_score','result']].to_sql(
    'matches', conn, if_exists='replace', index=False
)

print(f"✅ {len(matches)} matches inserted into DB")
print(matches['result'].value_counts())

✅ 380 matches inserted into DB
result
home_win    183
away_win    105
draw         92
Name: count, dtype: int64


# Step 8: Pull ALL 380 match events → store in DB


In [9]:
from tqdm import tqdm
import pandas as pd

all_events = []
failed_matches = []

for match_id in tqdm(matches['match_id'], desc="Pulling events"):
    try:
        ev = sb.events(match_id=match_id)
        ev['match_id'] = match_id
        all_events.append(ev)
    except Exception as e:
        failed_matches.append(match_id)

print(f"\n✅ Successfully pulled: {len(all_events)} matches")
print(f"❌ Failed: {len(failed_matches)} matches")

# Combine all events
events_df = pd.concat(all_events, ignore_index=True)
print(f"Total events: {len(events_df)}")
print(events_df['type'].value_counts().head(10))

Pulling events:   0%|          | 0/380 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
Pulling events:   0%|          | 1/380 [00:00<01:18,  4.85it/s]/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
Pulling events:   1%|          | 2/380 [00:00<02:23,  2.63it/s]/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
Pulling events:   1%|          | 3/380 [00:01<04:12,  1.49it/s]/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
Pulling events:   1%|          | 4/380 [00:02<03:48,  1.64it/s]/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoA


✅ Successfully pulled: 380 matches
❌ Failed: 0 matches
Total events: 1295354
type
Pass             365288
Ball Receipt*    328953
Carry            276171
Pressure         114081
Ball Recovery     38514
Duel              31578
Clearance         16217
Dribble           14550
Block             13916
Interception      12374
Name: count, dtype: int64


# Step 9: Clean & insert into events table


In [10]:
# Extract clean columns
def extract_outcome(x):
    try:
        return x.get('name', None) if isinstance(x, dict) else None
    except:
        return None

def extract_location(loc, idx):
    try:
        return loc[idx] if isinstance(loc, list) else None
    except:
        return None

events_clean = pd.DataFrame({
    'event_id':    events_df['id'],
    'match_id':    events_df['match_id'],
    'player_id':   events_df['player'].apply(lambda x: x.get('id') if isinstance(x, dict) else None),
    'player_name': events_df['player'].apply(lambda x: x.get('name') if isinstance(x, dict) else None),
    'team':        events_df['team'].apply(lambda x: x.get('name') if isinstance(x, dict) else None),
    'event_type':  events_df['type'].apply(lambda x: x.get('name') if isinstance(x, dict) else None),
    'minute':      events_df['minute'],
    'second':      events_df['second'],
    'location_x':  events_df['location'].apply(lambda x: extract_location(x, 0)),
    'location_y':  events_df['location'].apply(lambda x: extract_location(x, 1)),
    'outcome':     events_df.get('outcome', pd.Series([None]*len(events_df))).apply(extract_outcome)
})

# Insert to DB
events_clean.to_sql('events', conn, if_exists='replace', index=False)
print(f"✅ {len(events_clean)} events inserted into DB!")

✅ 1295354 events inserted into DB!


In [11]:
# First check what outcome columns exist
outcome_cols = [c for c in events_df.columns if 'outcome' in c.lower()]
print("Outcome columns:", outcome_cols)

# Safe player extractor
def safe_player(x, key):
    if isinstance(x, dict):
        return x.get(key, None)
    return None

# Smart outcome: check shot_outcome, pass_outcome etc.
def get_outcome(row):
    for col in ['shot_outcome', 'pass_outcome', 'dribble_outcome', 'duel_outcome']:
        if col in row.index and pd.notna(row.get(col)):
            val = row[col]
            if isinstance(val, dict):
                return val.get('name', None)
            elif isinstance(val, str):
                return val
    return None

events_clean = pd.DataFrame({
    'event_id':    events_df['id'],
    'match_id':    events_df['match_id'],
    'player_id':   events_df['player'].apply(lambda x: safe_player(x, 'id')),
    'player_name': events_df['player'].apply(lambda x: safe_player(x, 'name')),
    'team':        events_df['team'],        # already string ✅
    'event_type':  events_df['type'],        # already string ✅
    'minute':      events_df['minute'],
    'second':      events_df['second'],
    'location_x':  events_df['location'].apply(lambda x: x[0] if isinstance(x, list) else None),
    'location_y':  events_df['location'].apply(lambda x: x[1] if isinstance(x, list) else None),
    'outcome':     events_df.apply(get_outcome, axis=1)
})

# Re-insert
events_clean.to_sql('events', conn, if_exists='replace', index=False)
print(f"✅ {len(events_clean)} events re-inserted!")
print("\nEvent types:")
print(events_clean['event_type'].value_counts().head(8))
print("\nOutcome sample:")
print(events_clean['outcome'].value_counts().head(8))

Outcome columns: ['ball_receipt_outcome', 'dribble_outcome', 'duel_outcome', 'goalkeeper_outcome', 'interception_outcome', 'pass_outcome', 'shot_outcome', 'substitution_outcome', 'substitution_outcome_id']
✅ 1295354 events re-inserted!

Event types:
event_type
Pass             365288
Ball Receipt*    328953
Carry            276171
Pressure         114081
Ball Recovery     38514
Duel              31578
Clearance         16217
Dribble           14550
Name: count, dtype: int64

Outcome sample:
outcome
Incomplete         82715
Complete            8026
Out                 7601
Success In Play     4971
Won                 3842
Off T               2981
Lost In Play        2920
Lost Out            2828
Name: count, dtype: int64


# Step 10 : Match-level features

In [12]:
# Shots & Goals
shots = pd.read_sql("""
    SELECT match_id, team,
           COUNT(*) as total_shots,
           SUM(CASE WHEN outcome = 'Goal' THEN 1 ELSE 0 END) as goals,
           SUM(CASE WHEN outcome IN ('Saved', 'Goal', 'Off T', 'Blocked') THEN 1 ELSE 0 END) as shots_on_target
    FROM events
    WHERE event_type = 'Shot'
    GROUP BY match_id, team
""", conn)

# Pressing intensity
pressure = pd.read_sql("""
    SELECT match_id, team, COUNT(*) as total_pressures
    FROM events
    WHERE event_type = 'Pressure'
    GROUP BY match_id, team
""", conn)

# Pass accuracy (Complete = null outcome in statsbomb means successful)
passes = pd.read_sql("""
    SELECT match_id, team,
           COUNT(*) as total_passes,
           SUM(CASE WHEN outcome = 'Complete' THEN 1 ELSE 0 END) as completed_passes
    FROM events
    WHERE event_type = 'Pass'
    GROUP BY match_id, team
""", conn)

print("✅ Match features extracted!")
print(f"Shots: {len(shots)} | Pressure: {len(pressure)} | Passes: {len(passes)}")
print("\nShots sample:")
print(shots.head(5))
print("\nTop scoring teams:")
print(shots.groupby('team')['goals'].sum().sort_values(ascending=False).head(5))

✅ Match features extracted!
Shots: 759 | Pressure: 760 | Passes: 760

Shots sample:
   match_id       team  total_shots  goals  shots_on_target
0    265839  Barcelona           15      2               14
1    265839    Sevilla           12      1               10
2    265894  Barcelona           11      2               11
3    265894     Málaga           14      1               11
4    265944  Barcelona           23      6               23

Top scoring teams:
team
Barcelona          109
Real Madrid        108
Atlético Madrid     62
Athletic Club       57
Rayo Vallecano      52
Name: goals, dtype: int64


In [13]:
def get_outcome(row):
    for col in ['shot_outcome', 'pass_outcome', 'dribble_outcome', 'duel_outcome']:
        if col in row.index and pd.notna(row.get(col)):
            val = row[col]
            if isinstance(val, dict):
                return val.get('name', None)
            elif isinstance(val, str):
                return val
    return None

events_clean = pd.DataFrame({
    'event_id':    events_df['id'],
    'match_id':    events_df['match_id'],
    'player_id':   events_df['player_id'],   # ✅ direct column
    'player_name': events_df['player'],       # ✅ already string
    'team':        events_df['team'],         # ✅ already string
    'event_type':  events_df['type'],         # ✅ already string
    'minute':      events_df['minute'],
    'second':      events_df['second'],
    'location_x':  events_df['location'].apply(lambda x: x[0] if isinstance(x, list) else None),
    'location_y':  events_df['location'].apply(lambda x: x[1] if isinstance(x, list) else None),
    'outcome':     events_df.apply(get_outcome, axis=1)
})

events_clean.to_sql('events', conn, if_exists='replace', index=False)
print(f"✅ {len(events_clean)} events re-inserted!")
print("\nPlayer name sample:")
print(events_clean[events_clean['player_name'].notna()]['player_name'].head(5).tolist())
print("\nEvent types:")
print(events_clean['event_type'].value_counts().head(5))

✅ 1295354 events re-inserted!

Player name sample:
['Adrián González Morales', 'Borja González Tomás', 'Daniel García Carrillo', 'Aleksandar Pantić', 'Mauro Javier Dos Santos']

Event types:
event_type
Pass             365288
Ball Receipt*    328953
Carry            276171
Pressure         114081
Ball Recovery     38514
Name: count, dtype: int64


# Step 11: Module 2 — Player Ratings (Per 90 mins)


In [14]:
# Get total minutes played proxy (appearances per match)
player_minutes = pd.read_sql("""
    SELECT player_name, team,
           COUNT(DISTINCT match_id) as matches_played
    FROM events
    WHERE player_name IS NOT NULL
    GROUP BY player_name, team
""", conn)

# Attacking contributions
attacking = pd.read_sql("""
    SELECT player_name, team,
           SUM(CASE WHEN event_type = 'Shot' AND outcome = 'Goal' THEN 1 ELSE 0 END) as goals,
           SUM(CASE WHEN event_type = 'Shot' THEN 1 ELSE 0 END) as total_shots,
           SUM(CASE WHEN event_type = 'Dribble' AND outcome = 'Complete' THEN 1 ELSE 0 END) as dribbles_won
    FROM events
    WHERE player_name IS NOT NULL
    GROUP BY player_name, team
""", conn)

# Defensive contributions
defensive = pd.read_sql("""
    SELECT player_name, team,
           SUM(CASE WHEN event_type = 'Pressure' THEN 1 ELSE 0 END) as pressures,
           SUM(CASE WHEN event_type = 'Interception' THEN 1 ELSE 0 END) as interceptions,
           SUM(CASE WHEN event_type = 'Block' THEN 1 ELSE 0 END) as blocks,
           SUM(CASE WHEN event_type = 'Clearance' THEN 1 ELSE 0 END) as clearances
    FROM events
    WHERE player_name IS NOT NULL
    GROUP BY player_name, team
""", conn)

# Passing contributions
passing = pd.read_sql("""
    SELECT player_name, team,
           COUNT(*) as total_passes,
           SUM(CASE WHEN outcome = 'Complete' THEN 1 ELSE 0 END) as completed_passes
    FROM events
    WHERE event_type = 'Pass' AND player_name IS NOT NULL
    GROUP BY player_name, team
""", conn)
passing['pass_accuracy'] = (passing['completed_passes'] / passing['total_passes'] * 100).round(2)

print("✅ Player feature tables ready!")
print(f"Players tracked: {len(player_minutes)}")

✅ Player feature tables ready!
Players tracked: 546


In [21]:
# Fix pass accuracy query
passes_fixed = pd.read_sql("""
    SELECT match_id, team,
           COUNT(*) as total_passes,
           SUM(CASE WHEN outcome IS NULL THEN 1 ELSE 0 END) as completed_passes
    FROM events
    WHERE event_type = 'Pass'
    GROUP BY match_id, team
""", conn)

passes_fixed['total_passes'] = pd.to_numeric(passes_fixed['total_passes'], errors='coerce')
passes_fixed['completed_passes'] = pd.to_numeric(passes_fixed['completed_passes'], errors='coerce')
passes_fixed['pass_accuracy'] = (passes_fixed['completed_passes'] / passes_fixed['total_passes'] * 100).round(2)

print("✅ Fixed pass accuracy!")
print(passes_fixed.head())
print("\nPass accuracy stats:")
print(passes_fixed['pass_accuracy'].describe())

# Save fixed CSV
passes_fixed.to_csv('/kaggle/working/clubiq_data/match_passes.csv', index=False)
print("\n✅ match_passes.csv updated!")

✅ Fixed pass accuracy!
   match_id       team  total_passes  completed_passes  pass_accuracy
0    265839  Barcelona           720               610          84.72
1    265839    Sevilla           339               232          68.44
2    265894  Barcelona           597               504          84.42
3    265894     Málaga           314               217          69.11
4    265944  Barcelona           744               648          87.10

Pass accuracy stats:
count    760.000000
mean      74.487513
std        6.904439
min       52.150000
25%       69.777500
50%       74.680000
75%       79.197500
max       92.510000
Name: pass_accuracy, dtype: float64

✅ match_passes.csv updated!


# Step 12: Build Composite Player Rating


In [15]:
# Merge all features
ratings = player_minutes \
    .merge(attacking, on=['player_name','team'], how='left') \
    .merge(defensive, on=['player_name','team'], how='left') \
    .merge(passing[['player_name','team','total_passes','pass_accuracy']], on=['player_name','team'], how='left') \
    .fillna(0)

# Composite rating formula (weighted)
ratings['rating'] = (
    ratings['goals']         * 8.0 +
    ratings['total_shots']   * 0.5 +
    ratings['dribbles_won']  * 1.5 +
    ratings['pressures']     * 0.3 +
    ratings['interceptions'] * 2.0 +
    ratings['blocks']        * 1.5 +
    ratings['clearances']    * 1.0 +
    ratings['pass_accuracy'] * 0.2
).round(2)

# Normalize to 100
max_r = ratings['rating'].max()
ratings['rating_100'] = (ratings['rating'] / max_r * 100).round(2)

# Top 10 players
print("\n🏆 Top 10 Players — ClubIQ Rating:")
print(ratings[['player_name','team','goals','pass_accuracy','rating_100']]
      .sort_values('rating_100', ascending=False)
      .head(10).to_string(index=False))


🏆 Top 10 Players — ClubIQ Rating:
                   player_name                   team  goals  pass_accuracy  rating_100
 Neymar da Silva Santos Junior              Barcelona     24            0.0      100.00
   Asier Illarramendi Andonegi          Real Sociedad      1            0.0       95.25
             Antoine Griezmann        Atlético Madrid     22            0.0       95.21
             Gonzalo Escalante                  Eibar      3            0.0       94.42
Lionel Andrés Messi Cuccittini              Barcelona     26            0.0       92.28
      Luis Alberto Suárez Díaz              Barcelona     40            0.0       91.74
         Pedro Mosquera Parada RC Deportivo La Coruña      0            0.0       90.46
          Ander Capa Rodríguez                  Eibar      2            0.0       87.84
      Luis Hernández Rodríguez         Sporting Gijón      0            0.0       87.75
         Filipe Luís Kasmirski        Atlético Madrid      1            0.0       86.

In [16]:
# Module 3: Player Load Score (proxy from event intensity)
load = pd.read_sql("""
    SELECT player_name, team, match_id,
           COUNT(*) as total_actions,
           SUM(CASE WHEN event_type IN ('Pressure','Duel','Block','Clearance') THEN 1 ELSE 0 END) as high_intensity_actions,
           SUM(CASE WHEN event_type = 'Carry' THEN 1 ELSE 0 END) as carries,
           SUM(CASE WHEN event_type = 'Dribble' THEN 1 ELSE 0 END) as dribble_attempts
    FROM events
    WHERE player_name IS NOT NULL
    GROUP BY player_name, team, match_id
""", conn)

# Load score formula
load['load_score'] = (
    load['high_intensity_actions'] * 2.0 +
    load['carries']                * 0.5 +
    load['dribble_attempts']       * 1.0 +
    load['total_actions']          * 0.1
).round(2)

# Flag high load matches (top 10%)
threshold = load['load_score'].quantile(0.90)
load['high_load_flag'] = load['load_score'] >= threshold

# Per player summary
load_summary = load.groupby(['player_name','team']).agg(
    avg_load=('load_score','mean'),
    max_load=('load_score','max'),
    high_load_matches=('high_load_flag','sum'),
    total_matches=('match_id','count')
).round(2).reset_index()

load_summary['fatigue_risk'] = (load_summary['high_load_matches'] / load_summary['total_matches'] * 100).round(1)

print("✅ Load Monitoring Module ready!")
print(f"\n⚠️ Top 10 High Fatigue Risk Players:")
print(load_summary[load_summary['total_matches'] >= 10]
      .sort_values('fatigue_risk', ascending=False)
      [['player_name','team','avg_load','high_load_matches','total_matches','fatigue_risk']]
      .head(10).to_string(index=False))

✅ Load Monitoring Module ready!

⚠️ Top 10 High Fatigue Risk Players:
                     player_name                   team  avg_load  high_load_matches  total_matches  fatigue_risk
        Augusto Matías Fernández             Celta Vigo    120.60                 11             15          73.3
        Gabriel Fernández Arenas        Atlético Madrid    104.60                 25             35          71.4
           Pedro Mosquera Parada RC Deportivo La Coruña    105.36                 25             37          67.6
     Asier Illarramendi Andonegi          Real Sociedad    103.47                 22             33          66.7
             Daniel Parejo Muñoz               Valencia    106.79                 21             33          63.6
                      Toni Kroos            Real Madrid    106.32                 19             32          59.4
             Sergio Álvarez Díaz         Sporting Gijón    100.99                 16             27          59.3
Petros Matheus dos

#  save all our data to CSVs

In [17]:
import os
os.makedirs('/kaggle/working/clubiq_data', exist_ok=True)

# Save all module outputs
shots.to_csv('/kaggle/working/clubiq_data/match_shots.csv', index=False)
pressure.to_csv('/kaggle/working/clubiq_data/match_pressure.csv', index=False)
passes.to_csv('/kaggle/working/clubiq_data/match_passes.csv', index=False)
ratings.to_csv('/kaggle/working/clubiq_data/player_ratings.csv', index=False)
load_summary.to_csv('/kaggle/working/clubiq_data/load_summary.csv', index=False)
matches.to_csv('/kaggle/working/clubiq_data/matches.csv', index=False)

print("✅ All data exported!")
print(os.listdir('/kaggle/working/clubiq_data'))

✅ All data exported!
['match_pressure.csv', 'load_summary.csv', 'match_shots.csv', 'match_passes.csv', 'matches.csv', 'player_ratings.csv']


In [18]:
app_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

st.set_page_config(page_title="ClubIQ", page_icon="⚽", layout="wide")

# Load data
@st.cache_data
def load_data():
    shots    = pd.read_csv("match_shots.csv")
    pressure = pd.read_csv("match_pressure.csv")
    passes   = pd.read_csv("match_passes.csv")
    ratings  = pd.read_csv("player_ratings.csv")
    load     = pd.read_csv("load_summary.csv")
    matches  = pd.read_csv("matches.csv")
    return shots, pressure, passes, ratings, load, matches

shots, pressure, passes, ratings, load, matches = load_data()

# Sidebar
st.sidebar.image("https://img.icons8.com/emoji/96/soccer-ball-emoji.png", width=80)
st.sidebar.title("ClubIQ ⚽")
st.sidebar.markdown("**Full Stack Football Analytics**")
st.sidebar.markdown("---")
page = st.sidebar.radio("Navigate", ["🏟️ Match Intelligence", "🌟 Player Ratings", "💪 Load Monitoring"])

# ─── PAGE 1: MATCH INTELLIGENCE ───
if page == "🏟️ Match Intelligence":
    st.title("🏟️ Match Intelligence")
    st.markdown("Shooting efficiency, pressing intensity and pass accuracy across La Liga 2015/16")

    col1, col2, col3 = st.columns(3)
    col1.metric("Total Matches", len(matches))
    col2.metric("Total Goals", int(shots["goals"].sum()))
    col3.metric("Total Shots", int(shots["total_shots"].sum()))

    st.markdown("---")

    # Top teams by goals
    team_goals = shots.groupby("team")["goals"].sum().sort_values(ascending=False).reset_index().head(10)
    fig1 = px.bar(team_goals, x="team", y="goals", color="goals",
                  color_continuous_scale="Reds", title="Top 10 Teams by Goals Scored")
    st.plotly_chart(fig1, use_container_width=True)

    col4, col5 = st.columns(2)

    # Pressing intensity
    with col4:
        top_press = pressure.groupby("team")["total_pressures"].sum().sort_values(ascending=False).reset_index().head(10)
        fig2 = px.bar(top_press, x="team", y="total_pressures", color="total_pressures",
                      color_continuous_scale="Blues", title="Top 10 Teams by Pressing Intensity")
        st.plotly_chart(fig2, use_container_width=True)

    # Pass accuracy
    with col5:
        passes["pass_acc"] = (passes["completed_passes"] / passes["total_passes"] * 100).round(1)
        top_pass = passes.groupby("team")["pass_acc"].mean().sort_values(ascending=False).reset_index().head(10)
        fig3 = px.bar(top_pass, x="team", y="pass_acc", color="pass_acc",
                      color_continuous_scale="Greens", title="Top 10 Teams by Pass Accuracy (%)")
        st.plotly_chart(fig3, use_container_width=True)

# ─── PAGE 2: PLAYER RATINGS ───
elif page == "🌟 Player Ratings":
    st.title("🌟 Player Ratings")
    st.markdown("Composite ClubIQ rating based on goals, passing, dribbling, pressing and defensive actions")

    top_n = st.slider("Show Top N Players", 5, 50, 20)
    selected_team = st.selectbox("Filter by Team", ["All"] + sorted(ratings["team"].unique().tolist()))

    df = ratings.copy()
    if selected_team != "All":
        df = df[df["team"] == selected_team]
    df = df.sort_values("rating_100", ascending=False).head(top_n)

    fig4 = px.bar(df, x="rating_100", y="player_name", orientation="h",
                  color="rating_100", color_continuous_scale="Viridis",
                  hover_data=["team","goals","pass_accuracy"],
                  title=f"Top {top_n} Players — ClubIQ Rating")
    fig4.update_layout(yaxis={"categoryorder": "total ascending"}, height=600)
    st.plotly_chart(fig4, use_container_width=True)

    st.markdown("### 📋 Full Table")
    st.dataframe(df[["player_name","team","goals","total_shots","dribbles_won",
                      "pressures","interceptions","pass_accuracy","rating_100"]]
                 .reset_index(drop=True), use_container_width=True)

# ─── PAGE 3: LOAD MONITORING ───
elif page == "💪 Load Monitoring":
    st.title("💪 Load Monitoring")
    st.markdown("Player fatigue risk based on high-intensity actions per match")

    min_matches = st.slider("Minimum Matches Played", 5, 30, 10)
    df_load = load[load["total_matches"] >= min_matches].sort_values("fatigue_risk", ascending=False)

    col6, col7, col8 = st.columns(3)
    col6.metric("Players Monitored", len(df_load))
    col7.metric("Avg Load Score", round(df_load["avg_load"].mean(), 1))
    col8.metric("High Risk Players (>60%)", int((df_load["fatigue_risk"] > 60).sum()))

    st.markdown("---")

    fig5 = px.scatter(df_load.head(50), x="avg_load", y="fatigue_risk",
                      size="total_matches", color="fatigue_risk",
                      hover_name="player_name", hover_data=["team","total_matches"],
                      color_continuous_scale="RdYlGn_r",
                      title="Load vs Fatigue Risk (Top 50 Players)")
    st.plotly_chart(fig5, use_container_width=True)

    st.markdown("### ⚠️ High Risk Players")
    st.dataframe(df_load[df_load["fatigue_risk"] > 50]
                 [["player_name","team","avg_load","max_load","high_load_matches","total_matches","fatigue_risk"]]
                 .reset_index(drop=True), use_container_width=True)
'''

with open('/kaggle/working/app.py', 'w') as f:
    f.write(app_code)

print("✅ app.py created!")
print("Size:", len(app_code), "chars")

✅ app.py created!
Size: 5145 chars


In [19]:
print("🎯 ClubIQ Project Summary")
print("=" * 40)
print(f"✅ Matches analysed: {len(matches)}")
print(f"✅ Total events: 1,295,354")
print(f"✅ Players rated: {len(ratings)}")
print(f"✅ Top player: {ratings.sort_values('rating_100', ascending=False).iloc[0]['player_name']}")
print(f"✅ Highest fatigue risk: {load_summary.sort_values('fatigue_risk', ascending=False).iloc[0]['player_name']}")
print("=" * 40)
print("Modules: Match Intel | Player Ratings | Load Monitoring")
print("Stack: StatsBomb → Python → SQLite → Streamlit + Power BI + Tableau")

🎯 ClubIQ Project Summary
✅ Matches analysed: 380
✅ Total events: 1,295,354
✅ Players rated: 546
✅ Top player: Neymar da Silva Santos Junior
✅ Highest fatigue risk: Augusto Matías Fernández
Modules: Match Intel | Player Ratings | Load Monitoring
Stack: StatsBomb → Python → SQLite → Streamlit + Power BI + Tableau


In [20]:
# Check what's in match_passes.csv
passes_check = pd.read_csv('/kaggle/working/clubiq_data/match_passes.csv')
print(passes_check.head())
print(passes_check['completed_passes'].describe())

   match_id       team  total_passes  completed_passes
0    265839  Barcelona           720                 0
1    265839    Sevilla           339                 0
2    265894  Barcelona           597                 0
3    265894     Málaga           314                 0
4    265944  Barcelona           744                 0
count    760.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: completed_passes, dtype: float64
